In [2]:
import pandas as pd
from pathlib import Path
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Load cleaned data
df = pd.read_csv("../data/preprocess_data/complaints_cleaned.csv")

# Input and output
df["clean_text"] = df["Issue"].fillna("") + " " + df["Sub-issue"].fillna("")
X = df["clean_text"]
y = df["Product"]

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# TF-IDF
vectorizer = TfidfVectorizer(max_features=5000, stop_words="english")
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Train model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_vec, y_train)

model_dir = Path("../models/RandomForestRegressor")
model_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(rf_model, model_dir / "random_forest_model.joblib")

# Predict
y_pred_rf = rf_model.predict(X_test_vec)

# Evaluate
accuracy = accuracy_score(y_test, y_pred_rf)
report = str(classification_report(y_test, y_pred_rf, zero_division=0))
matrix = confusion_matrix(y_test, y_pred_rf)

print("Random Forest Accuracy:", accuracy)
print("\nClassification Report:\n", report)
print("\nConfusion Matrix:\n", matrix)

output_dir = Path("../results/Sristi")
output_dir.mkdir(parents=True, exist_ok=True)

with open(output_dir / "random_forest_evaluation_report.txt", "w", encoding="utf-8") as f:
    f.write(f"Random Forest Accuracy: {accuracy}\n\n")
    f.write("Classification Report:\n")
    f.write(report)
    f.write("\n\nConfusion Matrix:\n")
    f.write(str(matrix))


Random Forest Accuracy: 0.9777440724292416

Classification Report:
                                                                               precision    recall  f1-score   support

                                                     Bank account or service       1.00      1.00      1.00     17241
                                                 Checking or savings account       0.98      0.94      0.96      8128
                                                               Consumer Loan       0.95      0.98      0.96      6321
                                                                 Credit card       0.94      1.00      0.97     17838
                                                 Credit card or prepaid card       0.95      0.93      0.94      9530
                                                            Credit reporting       0.98      1.00      0.99     28086
Credit reporting, credit repair services, or other personal consumer reports       0.97      0.99      0.